In [ ]:
import numpy as np
import pandas as pd
from gams import transfer as gt
from pathlib import Path
import shutil

In [ ]:
import sys
sys.path.insert(0, snakemake.input.data2dd)
from data2dd_funcs import wrapdd, read_gen_database

In [ ]:
f_techno = snakemake.input.technoeconomic_database
f_scen = snakemake.input.scenario_db
gen_database_path = snakemake.input.gen_database
baseline_year = snakemake.params.baseline_year
zones = pd.read_csv(snakemake.input.zones).loc[:, "zone"]

psys_scen = snakemake.wildcards.psys_scenario
f_results = snakemake.input.results_gdx
gamspath = snakemake.params.gamspath

out_ledger = snakemake.output.ledger

In [ ]:
# reading trasnmission data
trans_sheet = pd.read_excel(f_techno, sheet_name="transmission", skiprows=1, engine="calamine")
trans_allowed = pd.read_excel(f_techno, sheet_name="transmission_allowed", skiprows=1, engine="calamine")


In [ ]:
planning_horizon = int(snakemake.params.planning_horizon)

In [ ]:
planning_horizon

In [ ]:
prior_ledger_path = snakemake.input.prior_ledger if snakemake.input.prior_ledger else None

# Update ledger 

In [ ]:
scen = pd.read_excel(f_scen, sheet_name="scenario_tech_definition", skiprows=0)
scen = scen.loc[(scen["Psys Scenario"] == psys_scen), :]
techs = scen["Technology Name (highRES)"]

In [ ]:
gen_params = (
    pd.read_excel(f_techno, sheet_name="gen", skiprows=1, engine="calamine")
)

In [ ]:
store_lifetime = (
    pd.read_excel(f_techno, sheet_name="store", skiprows=1, engine="calamine")
    .loc[:, ["Technology Name (highRES)", "e lifetime", "p lifetime"]]
    .rename(columns={"Technology Name (highRES)": "Technology"})
    .melt(id_vars="Technology", var_name="capacity_type", value_name="lifetime")
)

In [ ]:
store_lifetime["capacity_type"] = store_lifetime["capacity_type"].map(
    {"e lifetime": "ecap", "p lifetime": "pcap"}
)

In [ ]:
gen_lifetime = gen_params[["Technology Name (highRES)", "lifetime"]].rename(
    columns={"Technology Name (highRES)": "Technology"}
)

In [ ]:
trans_lifetime = trans_sheet[["Technology Name (highRES)", "lifetime"]].rename(
    columns={"Technology Name (highRES)": "Technology"}
)

In [ ]:
def melt_zone_wide(df, zones, value_name):
    m = df.melt(
        id_vars=["Technology", "parameter"],
        value_vars=list(zones),
        var_name="zone",
        value_name=value_name,
    )
    return m.dropna(subset=[value_name])

In [ ]:
def zledger(fin, sheet, techs, zones, tech_type, Y0):
    lim = pd.read_excel(fin, sheet_name=sheet, skiprows=0, engine="calamine")
    lim = lim.loc[(lim["Year"] == Y0) & (lim["Technology"].isin(techs)), :]

    cap = lim.loc[
        lim["parameter"].isin([tech_type + "_exist_pcap_z", tech_type + "_exist_ecap_z"]), :
    ]
    vintage = lim.loc[lim["parameter"] == tech_type + "_installed_year_z", :]

    cap_long = melt_zone_wide(cap, zones, "capacity_mw")
    cap_long["capacity_type"] = cap_long["parameter"].str.extract(r"_(pcap|ecap)_z")

    vintage_long = melt_zone_wide(vintage, zones, "installed_year")

    out = cap_long.merge(
        vintage_long[["Technology", "zone", "installed_year"]],
        on=["Technology", "zone"],
        how="left",
    )
    out["region"] = np.nan
    out["planning_horizon"] = Y0

    return out[
        ["Technology", "zone", "region", "capacity_type", "capacity_mw", "installed_year", "planning_horizon"]
    ]


In [ ]:
def tledger(trans_allowed, zones, Y0):
    # since transmission not dependent on psys scenarios
    lim = trans_allowed.loc[
        trans_allowed["Zone1"].isin(zones) & trans_allowed["Zone2"].isin(zones), :
    ].rename(columns={
        "Zone1": "zone",
        "Zone2": "region",
        "Tech": "Technology",
        "links_cap": "capacity_mw",
        "Installed year": "installed_year",
    })

    lim["capacity_type"] = "pcap"
    lim["planning_horizon"] = Y0

    return lim[
        ["Technology", "zone", "region", "capacity_type", "capacity_mw", "installed_year", "planning_horizon"]
    ]

In [ ]:

if prior_ledger_path is None:
    vre_techs = list(gen_params.loc[gen_params["set"] == "vre", "Technology Name (highRES)"])
    gen_seed = read_gen_database(gen_database_path, zones, vre_techs, None, baseline_year)
    gen_seed = gen_seed.loc[gen_seed["Technology"].isin(techs), :]

    z_only_store = techs[techs.isin(["PumpedHydro"])]
    z_store = zledger(f_techno, "store_exist_z", z_only_store, zones, "store", planning_horizon)

    t_gen = tledger(trans_allowed, zones, planning_horizon)

    gen_seed["installed_year"] = baseline_year
    gen_seed["planning_horizon"] = planning_horizon

    seed_ledger = pd.concat([gen_seed, z_store, t_gen], ignore_index=True)

    seed_ledger = seed_ledger.merge(gen_lifetime, on="Technology", how="left", suffixes=("", "_gen"))
    seed_ledger = seed_ledger.merge(
        store_lifetime, on=["Technology", "capacity_type"], how="left", suffixes=("", "_store")
    )
    seed_ledger = seed_ledger.merge(trans_lifetime, on="Technology", how="left", suffixes=("", "_trans"))
    seed_ledger["lifetime"] = seed_ledger["lifetime"].fillna(seed_ledger["lifetime_store"]).fillna(seed_ledger["lifetime_trans"])
    seed_ledger = seed_ledger.drop(columns=["lifetime_store", "lifetime_trans"])
else:
    seed_ledger = pd.read_csv(prior_ledger_path)

In [ ]:
gdx = gt.Container(f_results, system_directory=gamspath)

new_pcap_z = gdx.data["var_new_pcap_z"].records
new_vre_pcap_r = gdx.data["var_new_vre_pcap_r"].records
new_store_pcap_z = gdx.data["var_new_store_pcap_z"].records
new_store_ecap_z = gdx.data["var_new_store_ecap_z"].records
new_trans_pcap = gdx.data["var_new_trans_pcap"].records

In [ ]:
#TODO capacity units are assumed to be fixed to GW here, need to add flexibility

new_pcap_z["level"] = new_pcap_z["level"] * 1000
new_vre_pcap_r["level"] = new_vre_pcap_r["level"] * 1000
new_store_pcap_z["level"] = new_store_pcap_z["level"] * 1000
new_store_ecap_z["level"] = new_store_ecap_z["level"] * 1000
new_trans_pcap["level"] = new_trans_pcap["level"] * 1000

In [ ]:
new_vre = new_vre_pcap_r.loc[new_vre_pcap_r["level"] >= 0.001, :].rename(
    columns={"vre": "Technology", "z": "zone", "r": "region", "level": "capacity_mw"}
)
new_vre["capacity_type"] = "pcap"
new_vre["installed_year"] = planning_horizon
new_vre["planning_horizon"] = planning_horizon
new_vre = new_vre[
    ["Technology", "zone", "region", "capacity_type", "capacity_mw", "installed_year", "planning_horizon"]
]

vre_techs = new_vre["Technology"].unique()

In [ ]:
new_gen = new_pcap_z.loc[
    (new_pcap_z["level"] >= 0.001) & (~new_pcap_z["g"].isin(vre_techs)), :
].rename(columns={"g": "Technology", "z": "zone", "level": "capacity_mw"})
new_gen["region"] = np.nan
new_gen["capacity_type"] = "pcap"
new_gen["installed_year"] = planning_horizon
new_gen["planning_horizon"] = planning_horizon
new_gen = new_gen[
    ["Technology", "zone", "region", "capacity_type", "capacity_mw", "installed_year", "planning_horizon"]
]

In [ ]:
new_store_pcap = new_store_pcap_z.loc[new_store_pcap_z["level"] >= 0.001, :].rename(
    columns={"s": "Technology", "z": "zone", "level": "capacity_mw"}
)
new_store_pcap["region"] = np.nan
new_store_pcap["capacity_type"] = "pcap"
new_store_pcap["installed_year"] = planning_horizon
new_store_pcap["planning_horizon"] = planning_horizon
new_store_pcap = new_store_pcap[
    ["Technology", "zone", "region", "capacity_type", "capacity_mw", "installed_year", "planning_horizon"]
]

In [ ]:
new_store_ecap = new_store_ecap_z.loc[new_store_ecap_z["level"] >= 0.001, :].rename(
    columns={"s": "Technology", "z": "zone", "level": "capacity_mw"}
)
new_store_ecap["region"] = np.nan
new_store_ecap["capacity_type"] = "ecap"
new_store_ecap["installed_year"] = planning_horizon
new_store_ecap["planning_horizon"] = planning_horizon
new_store_ecap = new_store_ecap[
    ["Technology", "zone", "region", "capacity_type", "capacity_mw", "installed_year", "planning_horizon"]
]

In [ ]:
new_trans = new_trans_pcap.loc[new_trans_pcap["level"] >= 0.001, :].rename(
    columns={"z": "zone", "z_alias": "region", "trans": "Technology", "level": "capacity_mw"}
)

# var_new_trans_pcap comes back from GDX with both directions of every link populated
# Following drops the mirrored duplicate. capacity_mw always comes from the GDX

new_trans = new_trans.merge(
    trans_allowed[["Zone1", "Zone2", "Tech"]].rename(
        columns={"Zone1": "zone", "Zone2": "region", "Tech": "Technology"}
    ),
    on=["zone", "region", "Technology"],
    how="inner",
)

new_trans["capacity_type"] = "pcap"
new_trans["installed_year"] = planning_horizon
new_trans["planning_horizon"] = planning_horizon
new_trans = new_trans[
    ["Technology", "zone", "region", "capacity_type", "capacity_mw", "installed_year", "planning_horizon"]
]

In [ ]:
new_build = pd.concat([new_vre, new_gen, new_store_pcap, new_store_ecap, new_trans], ignore_index=True)

new_build = new_build.merge(gen_lifetime, on="Technology", how="left", suffixes=("", "_gen"))
new_build = new_build.merge(
    store_lifetime, on=["Technology", "capacity_type"], how="left", suffixes=("", "_store")
)
new_build = new_build.merge(trans_lifetime, on="Technology", how="left", suffixes=("", "_trans"))
new_build["lifetime"] = new_build["lifetime"].fillna(new_build["lifetime_store"]).fillna(new_build["lifetime_trans"])
new_build = new_build.drop(columns=["lifetime_store", "lifetime_trans"])

ledger = pd.concat([seed_ledger, new_build], ignore_index=True)

In [ ]:
ledger = ledger.sort_values(["installed_year", "Technology", "zone", "region"]).reset_index(drop=True)
ledger["installed_year"] = ledger["installed_year"].astype("Int64")

In [ ]:
ledger.to_csv(out_ledger, index=False)

In [ ]:
ledger

In [ ]:
# for future use to bring capacities update from technoeconomic rule
#area_out["key"] = area_out["Technology"] + "." + area_out["zone"] + "." + area_out["region"]

#wrapdd(area_out[["key", "area_km2"]].values, "area", "parameter", outfile=snakemake.output.vreareasdd)